In [ ]:
"""
rag_system.py

Complete RAG orchestration + prompt versioning + experiment logging.

Key features:
- Loads hierarchical JSON files produced earlier (json_data/dbe_<TICKER>/*.json).
- Creates LangChain Documents with rich metadata (ticker, file, section, subsection, table info).
- Builds vectorstore (FAISS) using OpenAI embeddings (configurable).
- Exposes multiple RAG strategies: naive, multi_query, fusion (LOTR/Merger), hyde, decomposition, step_back.
- LLM-based router chooses best strategy for a query (optional).
- Prompt registry & prompt versioning.
- Experiment logger: store prompt text, prompt version id, model name, config, retrieved doc IDs, LLM response.
- Evaluation helpers (EM/F1 and LLM-based evaluator).

Configure at top-of-file or pass a config object.
"""

import os
import re
import json
import uuid
import time
import logging
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from datetime import datetime

# LangChain imports
from langchain.chat_models import ChatOpenAI
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain.vectorstores.faiss import FAISS
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.prompts import PromptTemplate
from langchain.chains.base import Chain

# sklearn for TF-IDF sparse retriever
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import faiss

# ---------- Configuration ----------
class Config:
    # file paths
    JSON_INPUT_ROOT = Path("./json_data")        # expects json_data/dbe_<TICKER>/*.json
    VECTORSTORE_DIR = Path("./vectorstore_faiss")  # where FAISS index + mapping will be saved
    EXPERIMENTS_DIR = Path("./experiments")      # experiments logs
    PROMPTS_DIR = Path("./prompts")              # prompt versions registry

    # LLM settings
    llm_model = "gpt-4o-mini"   # your "OpenAI -4 mini" choice; override if needed
    llm_temperature = 0.0
    embedding_model_name = "text-embedding-3-small"  # or change as desired

    # retrieval
    k = 6  # number of docs to retrieve normally

    # logging
    LOG_LEVEL = logging.INFO

Config.VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
Config.EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
Config.PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=Config.LOG_LEVEL, format="%(asctime)s - %(levelname)s - %(message)s")

# ---------- Utilities: Prompt registry + Experiment Logger ----------

class PromptManager:
    """Simple local prompt/version registry."""
    def __init__(self, prompts_dir: Path):
        self.prompts_dir = prompts_dir
        self.prompts_dir.mkdir(parents=True, exist_ok=True)

    def register_prompt(self, name: str, template: str, description: str = "") -> str:
        """Create a new prompt version file; returns prompt_id."""
        pid = f"{name.replace(' ', '_')}_{int(time.time())}"
        data = {
            "id": pid,
            "name": name,
            "template": template,
            "description": description,
            "created_at": datetime.utcnow().isoformat()
        }
        path = self.prompts_dir / (pid + ".json")
        path.write_text(json.dumps(data, indent=2), encoding="utf-8")
        logging.info(f"Registered prompt {pid} -> {path}")
        return pid

    def load_prompt(self, prompt_id: str) -> Dict[str, Any]:
        p = self.prompts_dir / (prompt_id + ".json")
        if not p.exists():
            raise FileNotFoundError(f"Prompt id {prompt_id} not found at {p}")
        return json.loads(p.read_text(encoding="utf-8"))

    def list_prompts(self) -> List[str]:
        return [p.stem for p in sorted(self.prompts_dir.glob("*.json"))]

class ExperimentLogger:
    """Log experiments to JSONL files under experiments_dir + maintain a high-level index."""
    def __init__(self, experiments_dir: Path):
        self.dir = experiments_dir
        self.dir.mkdir(parents=True, exist_ok=True)
        self.index_path = self.dir / "index.jsonl"

    def log(self, entry: Dict[str, Any]) -> str:
        """Append entry (a dict) to a .jsonl file and return run_id."""
        run_id = f"run_{int(time.time()*1000)}_{uuid.uuid4().hex[:6]}"
        entry_with_meta = {
            "run_id": run_id,
            "timestamp": datetime.utcnow().isoformat(),
            **entry
        }
        # write one file and append to index
        run_path = self.dir / f"{run_id}.json"
        run_path.write_text(json.dumps(entry_with_meta, indent=2), encoding="utf-8")
        with self.index_path.open("a", encoding="utf-8") as idx:
            idx.write(json.dumps(entry_with_meta, ensure_ascii=False) + "\n")
        logging.info(f"Experiment logged: {run_id} -> {run_path}")
        return run_id

# ---------- Document ingestion: load hierarchical JSON → LangChain Documents ----------

def load_json_documents(json_root: Path) -> List[Document]:
    """
    Walk json_root/dbe_<TICKER>/*.json and convert hierarchical JSON into Documents.
    Each Document carries metadata about ticker, file, section, subsection, and table identities.
    """
    docs: List[Document] = []
    for dbe_dir in sorted(json_root.glob("dbe_*")):
        if not dbe_dir.is_dir():
            continue
        ticker = dbe_dir.name.replace("dbe_", "", 1)
        for jfile in sorted(dbe_dir.glob("*.json")):
            data = json.loads(jfile.read_text(encoding="utf-8"))
            file_name = data.get("file", jfile.name)
            # The JSON shape we wrote earlier: top-level "sections": [ {title, subsections, paragraphs, tables} ... ]
            sections = data.get("sections") or data.get("hierarchy") or []
            for sec_idx, sec in enumerate(sections):
                sec_title = sec.get("title") or sec.get("section") or ""
                # paragraphs attached directly to section
                for p_idx, para in enumerate(sec.get("paragraphs", [])):
                    content = para
                    metadata = {
                        "ticker": ticker,
                        "file": file_name,
                        "section_title": sec_title,
                        "section_index": sec_idx,
                        "subsection_title": None,
                        "paragraph_index": p_idx,
                        "is_table": False,
                        "source_path": str(jfile)
                    }
                    docs.append(Document(page_content=content, metadata=metadata))

                # subsections
                for sub_idx, sub in enumerate(sec.get("subsections", [])):
                    sub_title = sub.get("title") or sub.get("subsection") or ""
                    # paragraphs in subsection
                    for p_idx, para in enumerate(sub.get("paragraphs", [])):
                        content = para
                        metadata = {
                            "ticker": ticker,
                            "file": file_name,
                            "section_title": sec_title,
                            "section_index": sec_idx,
                            "subsection_title": sub_title,
                            "subsection_index": sub_idx,
                            "paragraph_index": p_idx,
                            "is_table": False,
                            "source_path": str(jfile)
                        }
                        docs.append(Document(page_content=content, metadata=metadata))

                    # tables in subsection
                    for t_idx, table in enumerate(sub.get("tables", [])):
                        # table_text: build a small text representation that maps header->value but with raw strings
                        table_title = table.get("title") or table.get("headers", []) and " | ".join(table.get("headers", [])) or "Table"
                        # Build a readable textual view for LLM synthesis (preserves raw values)
                        tbl_lines = [f"TABLE: {table_title}"]
                        # header labels
                        columns = table.get("columns") or []
                        # If columns are composite label dicts, use label
                        col_labels = [c.get("label") if isinstance(c, dict) else str(c) for c in columns] if columns else []
                        if col_labels:
                            tbl_lines.append("COLUMNS: " + " | ".join([str(l) for l in col_labels]))
                        # rows
                        for row in table.get("rows", []):
                            row_label = row.get("label") if isinstance(row, dict) else (row[0] if row else "")
                            cells = row.get("cells") if isinstance(row, dict) else {}
                            # cell mapping
                            cvpairs = []
                            for col_key, val in (cells or {}).items():
                                cvpairs.append(f"{col_key}: {val}")
                            tbl_lines.append(f"{row_label} -> " + "; ".join(cvpairs))
                        content = "\n".join(tbl_lines)
                        metadata = {
                            "ticker": ticker,
                            "file": file_name,
                            "section_title": sec_title,
                            "section_index": sec_idx,
                            "subsection_title": sub_title,
                            "subsection_index": sub_idx,
                            "table_index": t_idx,
                            "table_title": table_title,
                            "is_table": True,
                            "source_path": str(jfile)
                        }
                        docs.append(Document(page_content=content, metadata=metadata))

                # tables attached directly to section (if any)
                for t_idx, table in enumerate(sec.get("tables", [])):
                    table_title = table.get("title") or table.get("headers", []) and " | ".join(table.get("headers", [])) or "Table"
                    tbl_lines = [f"TABLE: {table_title}"]
                    columns = table.get("columns") or []
                    col_labels = [c.get("label") if isinstance(c, dict) else str(c) for c in columns] if columns else []
                    if col_labels:
                        tbl_lines.append("COLUMNS: " + " | ".join([str(l) for l in col_labels]))
                    for row in table.get("rows", []):
                        row_label = row.get("label") if isinstance(row, dict) else ""
                        cells = row.get("cells") if isinstance(row, dict) else {}
                        cvpairs = []
                        for col_key, val in (cells or {}).items():
                            cvpairs.append(f"{col_key}: {val}")
                        tbl_lines.append(f"{row_label} -> " + "; ".join(cvpairs))
                    content = "\n".join(tbl_lines)
                    metadata = {
                        "ticker": ticker,
                        "file": file_name,
                        "section_title": sec_title,
                        "section_index": sec_idx,
                        "subsection_title": None,
                        "table_index": t_idx,
                        "table_title": table_title,
                        "is_table": True,
                        "source_path": str(jfile)
                    }
                    docs.append(Document(page_content=content, metadata=metadata))
    logging.info(f"Loaded {len(docs)} documents from {json_root}")
    return docs

# ---------- Simple TF-IDF retriever (sparse) ----------

class TfidfRetriever:
    """A thin TF-IDF-based retriever over the provided Documents (returns langchain Documents)."""
    def __init__(self, documents: List[Document], k: int = 5):
        self.docs = documents
        self.k = k
        self.texts = [d.page_content for d in documents]
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=20000)
        self.mat = self.vectorizer.fit_transform(self.texts)  # sparse matrix (n_docs, n_feats)
        # store mapping
        self.docs_arr = documents

    def get_relevant_documents(self, query: str) -> List[Document]:
        v = self.vectorizer.transform([query])
        # compute dot products
        scores = (self.mat @ v.T).toarray().ravel()  # shape (n_docs,)
        top_idx = np.argsort(scores)[::-1][: self.k]
        result = [self.docs_arr[i] for i in top_idx if scores[i] > 0 or True]  # return top k even if zeros
        return result

# ---------- HyDE Retriever (generates hypothetical doc, then vector search) ----------

class HyDERetriever:
    """
    A HyDE-like retriever:
      - For each incoming query, ask an LLM to generate a hypothetical document (a concise answer-like paragraph).
      - Embed that hypothetical doc and search the vectorstore by vector similarity.
    """
    def __init__(self, llm: ChatOpenAI, embeddings: OpenAIEmbeddings, vectorstore: FAISS, k: int = 6):
        self.llm = llm
        self.emb = embeddings
        self.vs = vectorstore
        self.k = k
        self.hyde_prompt = PromptTemplate(
            input_variables=["question"],
            template=("Given the user question below, write a short hypothetical paragraph "
                      "that would likely contain the answer (a concise factual paragraph). "
                      "Do NOT include sources. Question: {question}")
        )

    def get_relevant_documents(self, question: str) -> List[Document]:
        # Generate hypothetical doc
        prompt = self.hyde_prompt.format(question=question)
        resp = self.llm(prompt) if callable(self.llm) else self.llm.generate([{"role":"user","content":prompt}])
        # For ChatOpenAI, resp is ChatOpenAI result - use .content if needed
        # For compatibility handle both simple call and object return
        hyde_text = None
        try:
            hyde_text = resp.content if hasattr(resp, "content") else (resp.generations[0][0].text if hasattr(resp, "generations") else str(resp))
        except Exception:
            hyde_text = str(resp)

        # embed hyde_text
        vector = self.emb.embed_query(hyde_text)
        docs = self.vs.similarity_search_by_vector(vector, k=self.k)
        return docs

# ---------- Decomposition Retriever (LLM splits into sub-queries) ----------

class DecompositionRetriever:
    """
    Ask LLM to create sub-questions; retrieve for each sub-question, union results.
    """
    def __init__(self, llm: ChatOpenAI, retriever, subq_prompt: Optional[PromptTemplate] = None, k_each: int = 4):
        self.llm = llm
        self.retriever = retriever
        self.k_each = k_each
        self.subq_prompt = subq_prompt or PromptTemplate(
            template=("Decompose the following question into 3-5 sub-questions that are easier to retrieve information for. "
                      "Return each sub-question on a separate line.\nQuestion: {question}"),
            input_variables=["question"]
        )

    def get_relevant_documents(self, question: str) -> List[Document]:
        prompt_text = self.subq_prompt.format(question=question)
        resp = self.llm(prompt_text)
        # extract text
        subq_text = None
        try:
            subq_text = resp.content if hasattr(resp, "content") else (resp.generations[0][0].text if hasattr(resp, "generations") else str(resp))
        except Exception:
            subq_text = str(resp)
        subqs = [s.strip("-. \t") for s in re.split(r"\\n|\\r", subq_text) if s.strip()]
        all_docs = []
        seen_ids = set()
        for sq in subqs:
            docs = self.retriever.get_relevant_documents(sq)
            for d in docs:
                # deduplicate by source path + content
                mid = (d.metadata.get("source_path"), d.page_content[:200])
                if mid not in seen_ids:
                    all_docs.append(d)
                    seen_ids.add(mid)
        return all_docs

# ---------- Build vectorstore (FAISS) ----------

def build_or_load_vectorstore(docs: List[Document], embeddings: OpenAIEmbeddings, vs_dir: Path) -> FAISS:
    """
    Build FAISS index from documents if not present; else load.
    """
    idx_file = vs_dir / "index.faiss"
    meta_file = vs_dir / "faiss_docstore.json"
    if idx_file.exists() and meta_file.exists():
        # load
        logging.info("Loading existing FAISS index from disk...")
        vs = FAISS.load_local(str(vs_dir), embeddings)
        return vs
    logging.info("Building FAISS index (this may take a few minutes for large corpora)...")
    vs = FAISS.from_documents(docs, embeddings)
    vs.save_local(str(vs_dir))
    logging.info(f"Saved FAISS index to {vs_dir}")
    return vs

# ---------- Answer generation (simple template + LLM) ----------

def answer_with_docs(llm: ChatOpenAI, docs: List[Document], question: str, prompt_template: str) -> Dict[str, Any]:
    """
    Given retrieved docs and a question, compose a prompt and ask LLM for an answer.
    Returns dict with answer text + provenance (list of source metadata).
    """
    # Compose context: include top-k documents (concise)
    ctxs = []
    for i, d in enumerate(docs):
        md = d.metadata or {}
        src = md.get("source_path", md.get("file", "unknown"))
        header = f"[Source {i+1}] {src} (section: {md.get('section_title')} / subsection: {md.get('subsection_title')})"
        ctxs.append(header + "\n" + d.page_content)

    context_block = "\n\n---\n\n".join(ctxs[: Config.k ])
    final_prompt = prompt_template.format(context=context_block, question=question)

    resp = llm(final_prompt)
    try:
        answer = resp.content if hasattr(resp, "content") else (resp.generations[0][0].text if hasattr(resp, "generations") else str(resp))
    except Exception:
        answer = str(resp)

    provenance = [{"source": (d.metadata.get("source_path") or d.metadata.get("file")), "metadata": d.metadata} for d in docs]
    return {"answer": answer, "provenance": provenance, "retrieved_count": len(docs)}

# ---------- Router: choose best RAG strategy ----------

def route_query(llm: ChatOpenAI, question: str, options: List[str]) -> str:
    """
    Ask LLM (temperature=0) to pick the best strategy from options.
    It should return the option string exactly. We keep the prompt simple and deterministic.
    """
    opt_lines = "\\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
    prompt = f"""You are a routing assistant. A user question is given below. Choose the single best retrieval strategy from the numbered list for answering this question (return only the option name exactly, not the number).

Question:
{question}

Strategies:
{opt_lines}

Return exactly one of the strategy names above (case-sensitive).
"""
    resp = llm(prompt)
    try:
        chosen = resp.content.strip() if hasattr(resp, "content") else resp.generations[0][0].text.strip()
    except Exception:
        chosen = str(resp).strip()
    # ensure valid
    if chosen not in options:
        # heuristic: pick first option that matches substring
        for o in options:
            if o.lower() in chosen.lower():
                return o
        return options[0]
    return chosen

# ---------- High-level RAG orchestrator ----------

class RAGSystem:
    def __init__(self, config: Config):
        self.cfg = config
        self.llm = ChatOpenAI(model_name=config.llm_model, temperature=config.llm_temperature)  # main model
        self.emb = OpenAIEmbeddings()  # uses environment OPENAI_API_KEY
        self.prompt_manager = PromptManager(config.PROMPTS_DIR)
        self.experiment_logger = ExperimentLogger(config.EXPERIMENTS_DIR)

        # placeholders to be built
        self.docs: List[Document] = []
        self.vs: Optional[FAISS] = None
        self.dense_retriever = None
        self.tfidf_retriever = None
        self.merger_retriever = None
        self.multiquery_retriever = None
        self.hyde_retriever = None
        self.decomp_retriever = None

    def ingest_documents(self):
        # load JSON → Documents
        self.docs = load_json_documents(self.cfg.JSON_INPUT_ROOT)
        # small safety: if no documents, raise
        if not self.docs:
            raise RuntimeError(f"No documents loaded from {self.cfg.JSON_INPUT_ROOT}")

    def build_indexes(self):
        # embeddings object already created
        self.vs = build_or_load_vectorstore(self.docs, self.emb, self.cfg.VECTORSTORE_DIR)
        # dense retriever
        self.dense_retriever = self.vs.as_retriever(search_type="similarity", search_kwargs={"k": self.cfg.k})
        # tfidf retriever
        self.tfidf_retriever = TfidfRetriever(self.docs, k=self.cfg.k)
        # merger retriever (LOTR)
        self.merger_retriever = MergerRetriever(retrievers=[self.dense_retriever, self.tfidf_retriever])
        # MultiQuery retriever (using llm to expand queries)
        self.multiquery_retriever = MultiQueryRetriever.from_llm(retriever=self.vs.as_retriever(), llm=self.llm)
        # HyDE
        self.hyde_retriever = HyDERetriever(self.llm, self.emb, self.vs, k=self.cfg.k)
        # Decomposition
        self.decomp_retriever = DecompositionRetriever(self.llm, self.vs.as_retriever())

    def run_query(self, question: str, strategy: Optional[str] = None,
                  prompt_id: Optional[str] = None, extra: Optional[Dict[str,Any]] = None) -> Dict[str, Any]:
        """
        Run the RAG pipeline for a single question using the requested strategy.
        strategy: one of ['naive','multi_query','fusion','hyde','decompose','step_back','all']
        If strategy is 'auto', we route using LLM router.
        prompt_id: prompt template id from PromptManager (if None, use default)
        """
        extra = extra or {}
        # determine prompt template
        if prompt_id:
            prompt_info = self.prompt_manager.load_prompt(prompt_id)
            prompt_template_str = prompt_info["template"]
            prompt_version = prompt_id
        else:
            prompt_template_str = ("You are an expert assistant. Use the context below to answer the question. "
                                   "Context:\n{context}\n\nQuestion: {question}\n\nAnswer with clear, concise, factual text. Cite sources in square brackets like [Source X].")
            prompt_version = "default"

        # route if needed
        strategies = ['naive','multi_query','fusion','hyde','decompose','step_back']
        if not strategy or strategy == "auto":
            chosen = route_query(self.llm, question, strategies)
            logging.info(f"Router selected strategy: {chosen}")
            strategy = chosen

        docs = []
        used_strategy = strategy

        # retrieval stage by strategy
        if strategy == "naive":
            docs = self.dense_retriever.get_relevant_documents(question)
        elif strategy == "multi_query":
            # MultiQueryRetriever returns union of docs
            docs = self.multiquery_retriever.get_relevant_documents(question)
        elif strategy == "fusion":
            docs = self.merger_retriever.get_relevant_documents(question)
        elif strategy == "hyde":
            docs = self.hyde_retriever.get_relevant_documents(question)
        elif strategy == "decompose":
            docs = self.decomp_retriever.get_relevant_documents(question)
        elif strategy == "step_back":
            # Step-back: initial answer then follow-up
            initial_docs = self.dense_retriever.get_relevant_documents(question)
            initial_answer = answer_with_docs(self.llm, initial_docs, question, prompt_template_str)
            # ask LLM what evidence is missing
            followup_prompt = (f"You produced the following answer:\n{initial_answer['answer']}\n\n"
                               f"Based on the answer and the question, generate up to 3 follow-up focused queries that would help obtain missing evidence. "
                               f"Return each on a new line.")
            followups_raw = self.llm(followup_prompt)
            try:
                followups_text = followups_raw.content if hasattr(followups_raw,"content") else followups_raw.generations[0][0].text
            except Exception:
                followups_text = str(followups_raw)
            followups = [l.strip("- .\\t") for l in re.split(r"\\n|\\r", followups_text) if l.strip()]
            extra_docs = []
            for fq in followups:
                extra_docs.extend(self.dense_retriever.get_relevant_documents(fq))
            # union + dedupe
            seen = set()
            docs = []
            for d in (initial_docs + extra_docs):
                key = (d.metadata.get("source_path"), d.page_content[:200])
                if key not in seen:
                    docs.append(d); seen.add(key)
            # final answer uses combined docs
        else:
            # default to naive
            docs = self.dense_retriever.get_relevant_documents(question)

        # answer generation
        ans = answer_with_docs(self.llm, docs, question, prompt_template_str)

        # log experiment
        log_entry = {
            "question": question,
            "strategy": used_strategy,
            "prompt_version": prompt_version,
            "prompt_template": prompt_template_str,
            "model_name": self.cfg.llm_model,
            "k_retrieved": len(docs),
            "retrieved_docs": [
                {"source": d.metadata.get("source_path") or d.metadata.get("file"),
                 "metadata": d.metadata}
                for d in docs
            ],
            "answer": ans["answer"]
        }
        run_id = self.experiment_logger.log(log_entry)
        return {"run_id": run_id, "answer": ans["answer"], "provenance": ans["provenance"]}

    # convenience wrapper
    def query_auto(self, question: str, prompt_id: Optional[str] = None):
        return self.run_query(question, strategy="auto", prompt_id=prompt_id)

# ---------- Evaluation helpers ----------

def exact_match(s1: str, s2: str) -> bool:
    return normalize_answer(s1) == normalize_answer(s2)

def normalize_answer(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return " ".join(s.split())

def f1_score(pred: str, gold: str) -> float:
    p_tokens = normalize_answer(pred).split()
    g_tokens = normalize_answer(gold).split()
    if not p_tokens or not g_tokens:
        return 0.0
    common = set(p_tokens) & set(g_tokens)
    if not common:
        return 0.0
    prec = len(common) / len(p_tokens)
    rec = len(common) / len(g_tokens)
    if prec+rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def evaluate_dataset(rag: RAGSystem, qa_pairs: List[Dict[str, str]], strategy: str = "auto", prompt_id: Optional[str] = None) -> Dict[str, Any]:
    """
    Run RAG on a list of {"question":..., "answer":...} and compute EM and F1 summary.
    """
    results = []
    for q in qa_pairs:
        out = rag.run_query(q["question"], strategy=strategy, prompt_id=prompt_id)
        pred = out["answer"]
        em = 1 if exact_match(pred, q["answer"]) else 0
        f1 = f1_score(pred, q["answer"])
        results.append({"question": q["question"], "gold": q["answer"], "pred": pred, "em": em, "f1": f1, "run_id": out["run_id"]})

    avg_em = sum(r["em"] for r in results) / len(results)
    avg_f1 = sum(r["f1"] for r in results) / len(results)
    return {"results": results, "avg_em": avg_em, "avg_f1": avg_f1}

# ---------- Usage sample (if run as script) ----------

def main_example():
    cfg = Config()
    rag = RAGSystem(cfg)
    rag.ingest_documents()
    rag.build_indexes()

    # create/register a prompt version
    default_prompt_template = ("You are an expert financial assistant. Use the following context to answer the user's question. "
                               "Cite sources as [Source X].\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:")
    prompt_id = rag.prompt_manager.register_prompt("financial_v1", default_prompt_template, "Default finance answer prompt")

    # sample query
    q = "What was MMM's Net interest expense in 2024 and 2023 according to the financials?"
    out = rag.run_query(q, strategy="auto", prompt_id=prompt_id)
    print("Run ID:", out["run_id"])
    print("Answer:", out["answer"])

if __name__ == "__main__":
    main_example()
